In [ ]:
!pip install ultralytics

In [ ]:
!unzip /content/UseableImages.zip

# **Data Preprocessing**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from PIL import Image

In [ ]:
!rm -rf '_MACOSX'

In [ ]:
!rm -rf '/content/UseableImages/.DS_Store'

In [ ]:
baseDir = "/content/UseableImages/"

for team in os.listdir(baseDir):
  teamDir = os.path.join(baseDir, team)
  i = 0
  for image in os.listdir(teamDir):
    imageDir = os.path.join(teamDir, image)
    os.rename(imageDir, f"{teamDir}/{team}_{i}.jpg")
    i += 1

# **YOLO Implementation**

In [ ]:
from ultralytics import YOLO
import cv2
import os

In [ ]:
model = YOLO('yolov8n.pt')

In [ ]:
baseDir = "/content/UseableImages/"
# Looping through every image to ensure there is no problem opening them after unzipping
for team in os.listdir(baseDir):
  teamDir = os.path.join(baseDir, team)
  for image in os.listdir(teamDir):
    imageDir = os.path.join(teamDir, image)
    try:
      img = Image.open(imageDir)
    except:
      print(f"Problem with file: {imageDir} - removing this file from dataset")
      os.remove(imageDir)

In [ ]:
def cropImages(inputDir, outputDir):
  # Make the new directory if it doesn't exist
  if not os.path.exists("CroppedImages"):
    os.mkdir("CroppedImages")
  # Looping through every team directory to crop the images
  for root, _, files in os.walk(inputDir):

    currentPath = os.path.relpath(root, inputDir)
    outputPath = os.path.join(outputDir, currentPath)
    # Making team directory in new output directory if it doesn't exist
    if not os.path.exists(outputPath):
      os.makedirs(outputPath)
    # Implementing an object detector on every image
    for file in files:

      imgPath = os.path.join(root, file)
      img = cv2.imread(imgPath)

      results = model(img)

      boxes = results[0].boxes.xyxy.cpu().numpy()
      # If one person is detected, then crop to that person
      if len(boxes) == 1:
        x1, y1, x2, y2 = boxes[0]
        croppedImg = img[int(y1):int(y2), int(x1):int(x2)]
        cv2.imwrite(os.path.join(outputPath, file), croppedImg)
      # Otherwise, crop an image for every single person detected in the image.
      elif len(boxes) > 1:
        largeBox = max(boxes, key=lambda box: (box[2]-box[0])*(box[3]-box[1]))
        x1, y1, x2, y2 = map(int, largeBox)
        croppedImg = img[int(y1):int(y2), int(x1):int(x2)]
        cv2.imwrite(os.path.join(outputPath, file), croppedImg)


inputBase = "/content/UseableImages"
outputBase = "/content/CroppedImages"

# Executing the function above on all the images in the UseableImages directory
cropImages(inputBase, outputBase)

In [ ]:
baseDir = "/content/CroppedImages/"
# Looping through every team and image in the team's directory and resizing them
for team in os.listdir(baseDir):
  teamDir = os.path.join(baseDir, team)
  for image in os.listdir(teamDir):
    imageDir = os.path.join(teamDir, image)
    try:
      img = Image.open(imageDir)
      image = img.resize((224, 224))
      image.save(imageDir)
    except:
      print(f"Problem with file: {imageDir} - removing this file from dataset")
      os.remove(imageDir)

# **PyTorch**

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torchvision.models as models

In [ ]:
# Creating the parameters for turning the images into a dataset ready for training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([.485, .456, .406], [.229, .224, .225])
    ])

# Turning the images in the CroppedImages directory into a dataset
imagePath="/content/CroppedImages"
dataset = ImageFolder(imagePath, transform=transform)
torch.save(dataset, 'nbaImageDataset1.pt')

# Creating an 80/20 training/test split
trainSplit = int(.8 * len(dataset))
testSplit = len(dataset) - trainSplit
trainDataset, testDataset = torch.utils.data.random_split(dataset, [trainSplit, testSplit])
trainLoader = DataLoader(trainDataset, batch_size=32, shuffle=True)
testLoader = DataLoader(testDataset, batch_size=32, shuffle=True)

In [ ]:
# Verifying cuda is available before training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
# Function for trianing the model, returning the trained model
def trainModel(model, epochs):
  for epoch in range(epochs):
    model.train()
    runningLoss = 0.0
    for images, labels in trainLoader:
      images, labels = images.to(device), labels.to(device)
      optimizer.zero_grad()
      outputs = model(images)
      loss = criterion(outputs, labels)
      loss.backward()
      optimizer.step()
      runningLoss += loss.item()
    epochLoss = runningLoss/len(trainLoader)
    print(f"Epoch: {epoch + 1}\tLoss: {epochLoss:.4f}")
  return model


# Function for evaluating the trained model on the test set
def evaluateModel(model, testLoader):

  model.to(device)
  model.eval()
  correct = 0
  total = 0
  # Testing predictions on test set generated earlier and returning the accuracy
  with torch.no_grad():
    for images, labels in testLoader:
      images, labels = images.to(device), labels.to(device)
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      total += labels.size(0)
      correct += (predicted == labels).sum().item()
  accuracy = 100 * correct / total
  return accuracy

**Mobilenet**

In [ ]:
# Mobilenet
model = models.mobilenet_v2()
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, 30)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = trainModel(model, 30)
print(f"Accuracy: {evaluateModel(model, testLoader):.4f}%")

**Resnet18 (Pretrained)**

In [ ]:
# Resnet18
model = models.resnet18(pretrained=True)
numFeatures = model.fc.in_features
model.fc = nn.Linear(numFeatures, 30)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = trainModel(model, 15)
print(f"Accuracy: {evaluateModel(model, testLoader):.4f}%")

**Resnet18**

In [ ]:
# Resnet18
model = models.resnet18(pretrained=False)
numFeatures = model.fc.in_features
model.fc = nn.Linear(numFeatures, 30)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = trainModel(model, 15)
print(f"Accuracy: {evaluateModel(model, testLoader):.4f}%")

**Resnet50**

In [ ]:
# Resnet50
model = models.resnet50()
numFeatures = model.fc.in_features
model.fc = nn.Linear(numFeatures, 30)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = trainModel(model, 15)
print(f"Accuracy: {evaluateModel(model, testLoader):.4f}%")

**Resnext50**

In [ ]:
# Resnext50
model = models.resnext50_32x4d(pretrained=True)
numFeatures = model.fc.in_features
model.fc = nn.Linear(numFeatures, 30)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

model = trainModel(model, 15)
print(f"Accuracy: {evaluateModel(model, testLoader):.4f}%")

In [ ]:
epochs = 50
runningLossList = []

model = models.resnet18()
numFeatures = model.fc.in_features
model.fc = nn.Linear(numFeatures, 30)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
# Searching for optimal # of epochs using resnet18
for epoch in range(epochs):
  model.train()
  runningLoss = 0.0
  for images, labels in trainLoader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    runningLoss += loss.item()
  epochLoss = runningLoss/len(trainLoader)
  runningLossList.append(epochLoss)
  print(f"Epoch: {epoch+1}\tLoss: {runningLoss/len(trainLoader):.4f}")

# Plotting Epochs
plt.plot(range(1, epochs + 1), runningLossList, linestyle='-')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")

In [ ]:
# Saving model
torch.save(model.state_dict(), 'model.pth')

# **END OF DEMO VIDEO**

# **Custom Model**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 53 * 53, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 30)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
for epoch in range(15):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, data in enumerate(trainLoader, 0):
        # get the inputs; data is a list of [inputs, labels]
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % 2000 == 1999:    # print every 2000 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 2000:.3f}')
            running_loss = 0.0



In [ ]:
accuracy = evaluateModel(net, testLoader)
print(f"Accuracy: {accuracy:.4f}%")

In [ ]:
plt.plot(x, accuracies, linestyle='-')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy")